# PHY408 Lab 01: Computational toolkit

**PHY408 Data-Intensive Science** · Xiamen University Malaysia  
Week 1, Lesson 2 (computer lab, 2 hours)  
**Classroom lesson this week (Lesson 1):** What is data-intensive science?  
**Data:** None required. Small examples are generated in the notebook; `inflammation-01.csv` is used if present, otherwise a synthetic stand-in is created.

> **Before you start:** work on a *copy* of this notebook inside your personal repository (File → Save As…), so that `git pull` on the course repository never conflicts with your edits.

> **Working offline:** everything except Part B.4 (GitHub) and the exit task runs without an internet connection. Cells starting with `%%bash` need a Unix shell, so run Jupyter from Linux, macOS or WSL (Windows users: launch Jupyter from the WSL terminal).

## Learning objectives

By the end of this lab you should be able to:

1. Navigate and manipulate files from the Unix shell, chain commands with pipes, and set file permissions
2. Track work with Git (the 3-state architecture, branching and merging) and push it to GitHub
3. Write basic Python: variables, lists, loops and functions, and test functions with `assert`
4. Use NumPy arrays and Pandas DataFrames for simple physics data

## Lab plan

| Block | Time |
| --- | --- |
| Recap | 5 min |
| Part A: Unix shell | 20 min |
| Part B: Git and GitHub | 25 min |
| Part C: Python essentials | 25 min |
| Part D: NumPy arrays | 20 min |
| Part E: Pandas DataFrames | 15 min |
| Exit task | 10 min |
| Open exercise | take-home |

## 0. Setup

Run the cell below. If an import fails, check that the `phy408` environment is active (see the course README).

In [ ]:
# --- Setup: run this cell first -------------------------------------------
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

rng = np.random.default_rng(seed=408)      # fixed seed -> reproducible results
OUT = Path("outputs"); OUT.mkdir(exist_ok=True)
DATA = Path("..") / "data"                 # shared course data folder

plt.rcParams.update({"figure.figsize": (7, 4.5), "axes.grid": True})
print("NumPy", np.__version__)
print("pandas", pd.__version__)

## 1. Recap

Scientific computing today rests on three layers, and this lab gives you one tool for each:

- **Environment (Unix):** the command line lets you navigate and control a machine without a GUI, which is how you will work on remote servers, clusters and the grid.
- **Version control (Git):** Git takes discrete snapshots of your work, so every result can be traced back to the exact code that produced it.
- **Collaboration (GitHub):** a shared remote copy of the repository lets many people build on the same analysis.

Python, NumPy and Pandas then sit on top of this foundation as the language you will use for every analysis in this course.

## Part A: Unix shell (~20 min)

**Goal:** navigate the directory tree, create and move files, chain commands with pipes, and set permissions.

Every command follows the same grammar, **command → flag → argument**:

```
ls   -l   /home/physics/data
│    │    └── argument: what data the command operates on
│    └─────── flag: how the command should behave (long format)
└──────────── command: the action (list)
```

The file system is an inverted tree starting at the root `/`. An **absolute path** starts from the root (`/home/user/project`); a **relative path** starts from where you are (`../` means one level up).

| Navigation and inspection | | Manipulation | |
| --- | --- | --- | --- |
| `pwd` | where am I? | `mkdir <dir>` | create a folder |
| `cd <path>` | move through the tree | `cp <src> <dest>` | copy a file |
| `ls -lah` | list everything, including hidden files | `mv <src> <dest>` | move or rename |
| `cat <file>` | print a file | `rm <file>` | delete (**no recycle bin!**) |

**Notebook tip:** a cell that starts with `%%bash` runs as a small shell script. Each `%%bash` cell starts a *fresh* shell in the notebook's folder, so every cell below begins with `cd`. All work happens in a scratch folder `~/phy408_sandbox`, well away from your real files.

In [ ]:
%%bash
# A.1 Navigation. The first two lines reset the scratch area (safe to re-run).
rm -rf ~/phy408_sandbox && mkdir -p ~/phy408_sandbox
cd ~/phy408_sandbox

pwd                              # where am I?
mkdir -p physics/data physics/analysis
ls                               # what is here?
ls -l physics                    # long format: permissions, owner, size, date
cd physics/data                  # relative jump down two branches
pwd
cd ../                           # relative jump up one branch
pwd
cd /                             # absolute jump to the root
ls

In [ ]:
%%bash
# A.2 Manipulation and redirection
cd ~/phy408_sandbox/physics

echo "run,energy_GeV" >  data/runs.csv      # >  creates (or overwrites!) a file
echo "1,91.2"         >> data/runs.csv      # >> appends to the end
echo "2,125.1"        >> data/runs.csv
cat data/runs.csv

cp data/runs.csv data/runs_backup.csv
mv data/runs_backup.csv analysis/old_runs.csv   # move and rename in one step
ls -lah data analysis

rm analysis/old_runs.csv                         # gone for good
ls analysis

**Pipes.** The `|` symbol sends the output (stdout) of one command into the input (stdin) of the next. This is the Unix philosophy: write programs that do one thing well, and make them work together.

In [ ]:
%%bash
# A.3 Pipeline: filter a detector log for signal events
cd ~/phy408_sandbox/physics/data

cat > data.txt << 'EOF'
[PHYSICS] Event 49580: calibration pulse
[PHYSICS] Event 49581: signal detected at 2.3 GeV
[PHYSICS] Event 49582: background noise
[PHYSICS] Event 49583: signal anomaly observed
[DAQ]     Event 49584: buffer full, event dropped
[PHYSICS] Event 49590: signal peak at 1.1 TeV
[PHYSICS] Event 49591: background noise
[PHYSICS] Event 49592: low-energy signal trace
EOF

echo "--- signal events ---"
cat data.txt | grep 'signal'
echo "--- how many? ---"
cat data.txt | grep 'signal' | wc -l     # chain a third tool to count lines

**Permissions.** `ls -l` shows a string such as `-rwxr-xr--`: three switches (read, write, execute) for the **owner**, the **group** and **others**. Each switch has a value, r = 4, w = 2, x = 1, and adding them gives one digit per class:

```
 rwx   r-x   r--
4+2+1 4+0+1 4+0+0
  7     5     4      →   chmod 754 script.py
```

A script needs the execute (`x`) bit before you can run it as `./script.py`.

In [ ]:
%%bash
# A.4 Permissions
cd ~/phy408_sandbox/physics/analysis

cat > script.py << 'EOF'
#!/usr/bin/env python3
print("Hello from an executable Python script!")
EOF

ls -l script.py                                   # -rw-r--r-- : no x bit
./script.py 2>/dev/null || echo ">>> Permission denied: no execute bit yet"
chmod 754 script.py
ls -l script.py                                   # -rwxr-xr--
./script.py

### Exercises (A)

**A1.** Create the folder `~/phy408_sandbox/muography/` with three subfolders `raw`, `processed` and `plots`. Inside `raw`, create `readme.txt` containing two lines, writing the first with `>` and the second with `>>`. Show the result with `ls -R ~/phy408_sandbox/muography` and `cat`.

In [ ]:
%%bash
# TODO (A1)


**A2.** Using `~/phy408_sandbox/physics/data/data.txt`:

1. print only the background lines;
2. count the lines that do **not** contain `signal` (hint: `grep -v`);
3. save the signal lines into a new file `signal_events.txt` and print it.

In [ ]:
%%bash
# TODO (A2)


**A3.** What numeric mode gives `-rwxr-x---`? Apply it to `script.py` with `chmod` and verify with `ls -l`. Who can now run the script, and who cannot even read it?

In [ ]:
%%bash
# TODO (A3)


**Check your understanding (A):** What is the difference between `>` and `>>`? What would happen to `runs.csv` if you had used `>` on every line in A.2?

*Your answer:* 

## Part B: Git and GitHub (~25 min)

**Goal:** init, add, commit, diff, branch and merge locally; then clone and push to GitHub.

**Git** is the version-control software that tracks changes on your machine. **GitHub** is a third-party website that hosts Git repositories so you can share and collaborate.

Git moves your files through three states:

```
 Working directory  ──git add──▶  Staging area  ──git commit -m '...'──▶  Local repository
 (your workbench:                (the drafting board:                     (the permanent ledger:
  files being edited)             changes bundled together)                hidden .git folder)
```

| Command | What it does |
| --- | --- |
| `git init` | create an empty repository (the `.git` folder) |
| `git status` | the most important command: shows where every file sits in the 3 states |
| `git diff` | line-by-line changes since the last commit |
| `git log` | print the history of snapshots |
| `git branch <name>` / `git checkout <name>` | start / switch to a parallel line of work |
| `git merge <name>` | fold a finished branch back into the current one |

Parts B.1 to B.3 work fully offline.

In [ ]:
%%bash
# B.0 Is Git installed, and does it know who you are?
git --version
echo "name : $(git config --global user.name  || echo '>>> NOT SET')"
echo "email: $(git config --global user.email || echo '>>> NOT SET')"

If either line says `NOT SET`, run these once **in your terminal** (use the email of your GitHub account):

```bash
git config --global user.name  "Your Name"
git config --global user.email "your_email@example.com"
```

In [ ]:
%%bash
# B.1 The 3-state architecture: untracked -> staged -> committed
cd ~/phy408_sandbox && rm -rf git_demo && mkdir git_demo && cd git_demo
git init -b main --quiet
# Fallback identity for this demo repository only, if none is set globally
git config user.name  > /dev/null || git config user.name  "PHY408 Student"
git config user.email > /dev/null || git config user.email "student@example.com"

echo 'print("Z boson mass = 91.19 GeV")' > analysis.py
echo "=== after creating the file ==="; git status --short   # ?? = untracked
git add analysis.py
echo "=== after git add ===";           git status --short   # A  = staged
git commit --quiet -m "start: first analysis script"
echo "=== after git commit ===";        git log --oneline

In [ ]:
%%bash
# B.2 See exactly what changed, then take a second snapshot
cd ~/phy408_sandbox/git_demo

echo 'print("Higgs mass = 125.1 GeV")' >> analysis.py
git diff                              # + lines are additions since the last commit
git add analysis.py
git commit --quiet -m "add Higgs mass"
git log --oneline

In [ ]:
%%bash
# B.3 Branching and merging: try an idea without touching main
cd ~/phy408_sandbox/git_demo

git branch feature-new-plot
git checkout --quiet feature-new-plot
echo 'print("plotting... (placeholder)")' > plot.py
git add plot.py
git commit --quiet -m "add plotting script"

git checkout --quiet main
echo "=== on main before merge ==="; ls         # plot.py lives only on the branch
git merge --quiet feature-new-plot
echo "=== on main after merge ===";  ls
git log --oneline --graph --all

### Exercises (B), offline

**B1.** In `~/phy408_sandbox/git_demo`, create a `README.md` describing the project in one line. Use `git status` before and after each step to watch it move through the three states, then commit it.

In [ ]:
%%bash
# TODO (B1)


**B2.** Create a branch `fix-units`, change the Z mass line in `analysis.py` to MeV (`91188 MeV`), commit on the branch, switch back to `main` and merge. Confirm with `git log --oneline --graph --all`, and paste that output below.

In [ ]:
%%bash
# TODO (B2)


### B.4 GitHub collaboration (online, in the terminal)

These steps talk to GitHub, so do them in a real terminal (not in `%%bash` cells, since `ssh-keygen` asks you questions interactively).

**Step 0: one-time SSH setup.** Register a GitHub account, then generate a key and add the *public* key (`.pub`) to GitHub → Settings → SSH and GPG keys.

```bash
ssh-keygen -t ed25519 -C "your_email@example.com"   # press Enter to accept the defaults
cat ~/.ssh/id_ed25519.pub                           # copy this line into GitHub
ssh -T git@github.com                               # test the connection
```

Success looks like: `Hi <username>! You've successfully authenticated, but GitHub does not provide shell access.`  
Guide: <https://docs.github.com/en/authentication/connecting-to-github-with-ssh/generating-a-new-ssh-key-and-adding-it-to-the-ssh-agent>

**Step 1: fork the course repository.** Go to the PHY408 repository on GitHub ([Instructor: PHY408_codes URL]) and click **Fork** to create your personal cloud copy.

**Step 2: bring it to your workbench.**

```bash
cd ~
git clone git@github.com:<your-username>/PHY408_codes.git
cd PHY408_codes
```

**Step 3: the full data-intensive loop.** Replace the fields with your own details:

```bash
echo 'No, NAME, ID, FavGam, DD/MM/YYYY' >> attendance.txt   # Unix modifies the data
git status                                                  # check the working directory
git add attendance.txt                                      # move to staging
git commit -m 'Add my name to attendance'                   # seal the snapshot
git push origin main                                        # sync back to GitHub
```

Open your fork in the browser and check that your line is there. You have just run the whole workflow: **Unix → Git → GitHub**.

**More practice:** <https://learngitbranching.js.org>

**TODO (B4):** paste the output of `git log --oneline` from your `PHY408_codes` clone here.

```
(paste here)
```

**Check your understanding (B):** Why should large data files not be committed?

*Your answer:* 

## Part C: Python essentials (~25 min)

**Goal:** variables, types, lists, loops and functions with physics quantities.

A **variable** is a label pointing to a value; re-assigning moves the label and the old value is gone. The three basic types are `int` (whole numbers), `float` (decimals) and `str` (text in quotes). Names may contain letters, digits and underscores, cannot start with a digit, and are case-sensitive.

In [ ]:
# C.1 Variables, types and built-in functions
n_particles = 1000          # int
mass_GeV    = 125.1         # float: Higgs boson mass
detector    = 'CMS'         # str
print(type(n_particles), type(mass_GeV), type(detector))

# Energy-momentum relation for an electron
m = 0.511                   # electron mass in MeV
p = 10.0                    # momentum in MeV/c
E = (p**2 + m**2)**0.5
print('Energy =', E, 'MeV')

# Other built-ins convert between types
print(len('hello'), int(3.7), float(42), str(100))

A **list** is an ordered, mutable collection. Python counts from **0**, and a slice `a[i:j]` runs from index `i` up to, *but not including*, `j`. The same indexing rules apply to strings and NumPy arrays.

In [ ]:
# C.2 Lists, indexing and slicing
detectors = ['ATLAS', 'CMS', 'LHCb', 'ALICE']
detectors.append('TOTEM')           # add to the end
print(len(detectors))
print(detectors[0], detectors[-1])  # first, last
print(detectors[1:3])               # ['CMS', 'LHCb']
print(detectors[:2], detectors[2:])

A **for loop** repeats its indented body once per item. Two patterns you will use constantly: the **accumulator** (build up a result) and `enumerate` (get index and value together). `range(a, b)` generates integers from `a` up to, but not including, `b`.

In [ ]:
# C.3 Loop patterns
lepton_masses = [0.511, 105.7, 1776.9]    # e, mu, tau in MeV

total = 0                                  # accumulator
for mass in lepton_masses:
    total = total + mass
print('Sum of lepton masses:', total, 'MeV')

for i, name in enumerate(['electron', 'muon', 'tau']):
    print(f'Particle {i}: {name}')

for i in range(1, 6):
    print(i, 'squared =', i**2)

### Exercises (C), loops

**C1.** Given `velocities = [3.0, 5.0, 8.0, 12.0]` (m/s) and `m = 2.0` kg, use a loop to print $KE = \tfrac{1}{2} m v^2$ for each velocity, e.g. `v = 3.0 m/s, KE = 9.0 J`.

In [ ]:
# TODO (C1)


**C2.** Use `range()` and the accumulator pattern to compute $1 + 2 + \dots + 100$. Finish with `assert total == 5050`.

In [ ]:
# TODO (C2)


**Functions** package reusable code: `def name(parameters):`, an indented body, and `return` to send back the result. Small, tested functions can then be **composed** into larger ones. Variables created inside a function are **local** and disappear when it returns, so keep a result by assigning the call. A **docstring** documents the function (read it with `help()`), and `assert` checks it against a known answer.

In [ ]:
# C.4 Defining, composing, documenting and testing functions
def fahr_to_celsius(temp):
    """Convert a temperature from Fahrenheit to Celsius."""
    return (temp - 32) * (5/9)

def celsius_to_kelvin(temp_c):
    """Convert a temperature from Celsius to Kelvin."""
    return temp_c + 273.15

def fahr_to_kelvin(temp_f):
    """Convert Fahrenheit to Kelvin by composing the two functions above."""
    temp_c = fahr_to_celsius(temp_f)
    return celsius_to_kelvin(temp_c)

result = fahr_to_kelvin(212.0)
print(result)                       # 373.15

help(fahr_to_celsius)               # shows the docstring

assert fahr_to_celsius(32) == 0.0
assert fahr_to_celsius(212) == 100.0
print('All tests passed!')

# Scope: temp_c existed only inside fahr_to_kelvin
try:
    print(temp_c)
except NameError as err:
    print('NameError:', err)

### Exercises (C), functions

**C3.** Write `kinetic_energy(mass, velocity)` that returns $\tfrac{1}{2} m v^2$, with a docstring. Test it with `assert kinetic_energy(2.0, 10.0) == 100.0`, then rewrite your C1 loop to call the function.

In [ ]:
# TODO (C3)


**C4.** Write `grav_force(m1, m2, r)` returning $F = G m_1 m_2 / r^2$ with $G = 6.674\times10^{-11}$ N m² kg⁻². For the Earth–Moon system ($m_1 = 5.97\times10^{24}$ kg, $m_2 = 7.35\times10^{22}$ kg, $r = 3.84\times10^{8}$ m) you should get about $1.98\times10^{20}$ N; print it with `f'{F:.2e}'`.

**C5 (extend it).** Add a docstring, then write an `assert` for a case whose answer you know independently: the force on a 1 kg mass at the Earth's surface ($R_\oplus = 6.371\times10^{6}$ m) should be close to $g \approx 9.8$ N. Floats are rarely *exactly* equal, so compare with `abs(a - b) < tol` or `np.isclose(a, b, rtol=...)` rather than `==`.

In [ ]:
# TODO (C4, C5)


**Check your understanding (C):** What does `type(125.1)` return, and why? Why is `==` safe in the `fahr_to_celsius` tests above but risky in general for floats?

*Your answer:* 

## Part D: NumPy arrays (~20 min)

**Goal:** vectorised calculations instead of loops.

A NumPy array holds numbers of one type in a compact block of memory, and arithmetic acts on every element at once, with no loop, typically 10–100× faster than a Python list.

We use the Software Carpentry **inflammation** dataset: 60 patients (rows, axis 0) × 40 days (columns, axis 1), each cell an inflammation score. If `inflammation-01.csv` is not in the course `data/` folder, the next cell builds a synthetic stand-in with the same shape so you can work offline. (The real file is in <https://github.com/SiewYan/python-novice-inflammation>.)

In [ ]:
# D.1 Lists vs arrays, and loading data
vals = [1, 2, 3, 4]
print([x * 2 for x in vals])        # list: needs a loop
arr = np.array(vals)
print(arr * 2)                      # array: vectorised

print(np.zeros((2, 3)), np.arange(0, 10, 2), np.linspace(0, 1, 5), sep='\n')

fname = DATA / "inflammation-01.csv"
if fname.exists():
    data = np.loadtxt(fname=fname, delimiter=",")
    print("Loaded", fname)
else:
    # Offline fallback: synthetic scores that rise and fall over 40 days
    days = np.arange(40)
    expected = np.minimum(days, 40 - days) / 2
    data = rng.poisson(lam=np.tile(expected, (60, 1))).astype(float)
    print("inflammation-01.csv not found -> using a synthetic stand-in")

In [ ]:
# D.2 Attributes and slicing
print(type(data))
print('shape:', data.shape, ' dtype:', data.dtype, ' ndim:', data.ndim, ' size:', data.size)

print(data[0, 0])        # first patient, first day
print(data[29, 19])      # patient 30, day 20
print(data[0:4, 0:10])   # first 4 patients, first 10 days
print(data[:, 0])        # day 1 for all patients (':' means "all along this axis")

**The axis rule:** `axis=0` collapses the rows (one value per column, i.e. per day); `axis=1` collapses the columns (one value per row, i.e. per patient).

In [ ]:
# D.3 Aggregations along an axis, and a first plot
print('overall mean:', np.mean(data), ' max:', np.max(data), ' std:', np.std(data))

daily_mean  = np.mean(data, axis=0)     # shape (40,)
daily_max   = np.amax(data, axis=0)
daily_min   = np.amin(data, axis=0)
patient_avg = np.mean(data, axis=1)     # shape (60,)
print(daily_mean.shape, patient_avg.shape)

fig, axes = plt.subplots(1, 3, figsize=(10, 3.5), sharex=True)
for ax, values, label in zip(axes, [daily_mean, daily_max, daily_min], ['mean', 'max', 'min']):
    ax.plot(values)
    ax.set_xlabel('day')
    ax.set_ylabel(f'daily {label}')
fig.tight_layout()
fig.savefig(OUT / "lab01_inflammation_daily.pdf")
plt.show()

### Exercises (D)

**D1.** Which patient has the highest average inflammation, and on which day is the daily mean largest? (Hint: `np.argmax`.) Remember to report them in human counting, starting from 1.

In [ ]:
# TODO (D1)


**D2.** Put the "10–100× faster" claim to the test. The cell below generates momentum components for $10^6$ particles. Compute $|\vec p| = \sqrt{p_x^2 + p_y^2 + p_z^2}$ (a) with a Python `for` loop and (b) with one vectorised NumPy expression. Time both with `time.perf_counter()`, check that the answers agree with `np.allclose`, and print the speed-up.

In [ ]:
# TODO (D2)
N = 1_000_000
px, py, pz = rng.normal(loc=0.0, scale=10.0, size=(3, N))    # GeV/c

# (a) loop version

# (b) vectorised version

# (c) compare results and timings


**Check your understanding (D):** How much faster is the vectorised version on your machine? Does `data[5, :]` create a copy of the data or a view?

*Your answer:* 

## Part E: Pandas DataFrames (~15 min)

**Goal:** load, select and group tabular data.

A DataFrame is a table with labelled rows and columns, like a spreadsheet you can program. Pandas handles mixed types, missing values and CSV files naturally, and does filtering and grouping in one line.

| Selection | |
| --- | --- |
| `df['col']` | one column |
| `df.loc[row, col]` | by label |
| `df.iloc[i, j]` | by position |
| `df[df['x'] > 0]` | rows satisfying a condition |

**Group-by** follows *split → apply → combine*: split the rows by a key, apply an operation to each group, combine the results.

Our dataset is a small pseudo dataset of top-quark mass measurements (GeV).

In [ ]:
# E.1 Prepare the pseudo dataset, save it, and read it back
data_top = {
    "experiment":  ["ATLAS", "CMS", "CDF", "D0", "ATLAS", "CMS", "CDF", "D0"],
    "year":        [2018, 2019, 2014, 2016, 2021, 2022, 2012, 2015],
    "mass":        [172.51, 172.44, 173.20, 174.98, 172.69, 172.76, 173.18, 174.34],
    "uncertainty": [0.50, 0.48, 0.76, 0.76, 0.30, 0.29, 0.94, 0.64],
}
pd.DataFrame(data_top).to_csv(OUT / "top_quark_mass.csv", index=False)

df = pd.read_csv(OUT / "top_quark_mass.csv")
display(df.head())
df.info()

In [ ]:
# E.2 Selection, filtering and group-by
print(df['mass'].values)
print(df.loc[0, 'experiment'], df.iloc[0, 2])
display(df[df['year'] >= 2018])                       # LHC-era rows only
display(df.groupby('experiment')['mass'].mean())      # split -> apply -> combine

### Exercises (E)

**E1. Trend over time.** Sort the table by year. Does the uncertainty improve with time?

**E2. Best measurement.** Which row is the most precise? (Hint: `df.loc[df['uncertainty'].idxmin()]`.)

**E3. Summary statistics.** Print the mean, standard deviation, minimum and maximum of `mass`.

In [ ]:
# TODO (E1, E2, E3)


**E4. Simple outlier detection.** Using the interquartile range $\mathrm{IQR} = Q_3 - Q_1$ (from `df['mass'].quantile(...)`), flag measurements below $Q_1 - 1.5\,\mathrm{IQR}$ or above $Q_3 + 1.5\,\mathrm{IQR}$. Are any flagged? Would you expect this rule to be reliable with only 8 points?

In [ ]:
# TODO (E4)


**E5. Combine the measurements.** Compute the inverse-variance weighted mean

$$\bar m = \frac{\sum_i m_i/\sigma_i^2}{\sum_i 1/\sigma_i^2}, \qquad \sigma_{\bar m} = \left(\sum_i 1/\sigma_i^2\right)^{-1/2},$$

and compare it with the plain mean from E3. Which measurements dominate the weighted mean, and why? (You will meet this formula again in the final project.)

**E6. Plot.** Plot `mass` against `year` with error bars (`plt.errorbar`), one colour per experiment, draw the weighted mean as a horizontal line, label the axes with units, and save the figure as `outputs/lab01_main_figure.pdf`.

In [ ]:
# TODO (E5, E6)


**Check your understanding (E):** When would you use Pandas instead of a NumPy array?

*Your answer:* 

## Exit task (~10 min)

1. Make sure every cell runs from top to bottom (Kernel → Restart & Run All).
2. Save your key figure(s) to `outputs/`.
3. Push this notebook and `outputs/` to your personal GitHub repository.
4. Submit the commit SHA (`git log -1 --format=%H`) on Moodle.

In [ ]:
# Exit task: commit and push from the terminal (not from this cell):
#   git add lab01_*.ipynb outputs/
#   git commit -m "Lab 01"
#   git push

## Open exercise (take-home)

Create a small CSV of measurements of your own choice **from the shell** (using `echo` with `>` and `>>`), load it with Pandas and make one labelled plot. Commit the CSV, notebook and figure with meaningful commit messages.

There is no single correct answer. Explain your choices in markdown cells.

In [ ]:
# Open exercise: your work here


## Further reading

- L. Lista, *Statistical Methods for Data Analysis in Particle Physics*, 3rd ed. (2023)
- Software Carpentry, *The Unix Shell*: <https://swcarpentry.github.io/shell-novice/>
- Software Carpentry, *Version Control with Git*: <https://swcarpentry.github.io/git-novice/>
- Software Carpentry, *Programming with Python* (inflammation data): <https://swcarpentry.github.io/python-novice-inflammation/>
- Learn Git Branching (interactive): <https://learngitbranching.js.org>
- *10 minutes to pandas*: <https://pandas.pydata.org/docs/user_guide/10min.html>